# 12 - The encoder without attack examples, on the adversarially trained BERT

**Question.** On a plain BERT, an encoder trained without attack examples scores below TMD (validation: 40.0 / 49.5 vs 50.5 / 57.0). Does that change when BERT is adversarially trained? This isolates the *encoder idea itself*: BERT is the adversarially trained one from notebook 07b (`advbert2`) in every row, and only the projection differs.

**The encoder here** is notebook 07b's `b_enc_den`. It is trained on the clean embeddings of `advbert2`, with the generic random-noise copies of notebook 07 as its denoising target. It sees no attacked sentence and no synonym list, so it is attack-free in the strict sense. (The earlier plain-BERT row used synonym-swapped embeddings for its denoising term; counter-fitted neighbours are also what TextFooler and PWWS draw from, so it was not strictly attack-free.)

**Setup.** The same 200 validation sentences as notebooks 06-08, TextFooler and BERT-Attack. Nothing touches the test set. Rows for `advbert2`, `advbert2_tmd` and `advbert2_enc1` are read from Drive (07b).

**Cost (L4):** about 15 minutes. Needs 07b's files on Drive.

In [ ]:
!pip -q install transformers datasets accelerate textattack tensorflow_hub
import os, sys, subprocess
REPO = 'https://github.com/shivanshgoyaliitk/for_shreyansh.git'
BRANCH = 'claude/magical-pasteur-ptp2k2'
DIR = '/content/for_shreyansh'
url = REPO
try:  # only needed if the repo is private: add a Colab secret named GITHUB_TOKEN
    from google.colab import userdata
    url = REPO.replace('https://', 'https://' + userdata.get('GITHUB_TOKEN') + '@')
except Exception:
    pass
if os.path.exists(DIR):
    subprocess.run(['git', '-C', DIR, 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, DIR], check=True)
sys.path.insert(0, DIR)

In [ ]:
import os, torch, tmd_lab as T
T.setup_env()                      # the authors' TextAttack fork for every attack
from google.colab import drive
drive.mount('/content/drive')

DATASET = 'ag_news'
WORK = '/content/drive/MyDrive/tmd_run_paperft'   # same folder as notebooks 00-11
N_VAL = 200
SCREEN_ATTACKS = ('textfooler', 'bertattack')

lab = T.Lab(DATASET, work=WORK)
lab.finetune('base', **T.PAPER_FT)
E_tr, Y_tr = lab.embeddings('train', 'base')
E_va, Y_va = lab.embeddings('val', 'base')
gan = lab.get_gan('base', E_tr, E_va)

lab.use_model('advbert2')           # notebook 07b's adversarially trained BERT
Ea_tr, _ = lab.embeddings('train', 'advbert2')
Ea_va, _ = lab.embeddings('val', 'advbert2')
gan_a = lab.get_gan('advbert2', Ea_tr, Ea_va)

path = lab._path(f'projenc_b_enc_den_L{gan_a.latent_dim}C{gan_a.code_dim}.pt')
assert os.path.exists(path), 'run notebook 07b first (it trains b_enc_den)'
enc_den = T.ProjEncoder(gan_a.latent_dim, gan_a.code_dim).to(T.DEV)
enc_den.load_state_dict(torch.load(path))
enc_den.eval()

In [ ]:
lab.use_model('advbert2')
v = T.Victim(lab, gan=gan_a, enc=enc_den, mode='tmd', proj_mode='encoder')
T.evaluate(lab, v, 'advbert2_encden', Ea_va, Y_va, attacks=SCREEN_ATTACKS, n=N_VAL, split='val', need_log=True)
tab = T.screen(lab, ['vanilla', 'baseline', 'advbert2', 'advbert2_tmd', 'advbert2_encden', 'advbert2_enc1'], N_VAL)
display(tab)

In [ ]:
row = tab.set_index('variant')
gap = row.loc['advbert2_encden', ['TextFooler', 'BERT-Attack']] - row.loc['advbert2_tmd', ['TextFooler', 'BERT-Attack']]
print('attack-free encoder minus TMD random search, both on the adversarially trained BERT (validation, 200 sentences):')
print(gap.round(1).to_string())
print('noise margin at 200 sentences is about +-7 points per column')